# 03 — Eval

Phase 3 (Per-Field-Accuracy Baseline), Phase 4 (Iterations-Auswertung + Synthese-Tabelle), Phase 6 (Skalierung 7B + 3B-Halluzinations-Klassen).

Predictions kommen aus `02_extract.ipynb`. Gold ist eure `annotation/meine_gold.csv` aus Phase 2.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | 2025-05-15 |
| Gold-Datei | `annotation/meine_gold.csv` |
| Aktive Predictions-Datei(en) | `predictions.jsonl`, `predictions_iter_A.jsonl`, `predictions_iter_B.jsonl` |
| Match-Entscheidung `skills_top3` | **Set-Match** (Reihenfolge egal) |
| Match-Entscheidung `gehalt_min_eur` | **Exakter Match** (null=null OK) |
| JSON-Parse-Fails (Anzahl) | 0 (Baseline) |

## Phase 3 — Baseline-Accuracy auf 12 Hand-Gold-Anzeigen

Baseline-Lauf mit dem Original-Prompt (Run-Tag baseline, predictions.jsonl) auf den 12 hand-annotierten Anzeigen. Die Tabelle unten zeigt die Accuracy pro Feld als Ausgangspunkt für die Iterationen in Phase 4.

In [1]:
import json
import pandas as pd
from pathlib import Path

SCHEMA_COLS = ['id', 'homeoffice', 'vertragsart', 'erfahrungslevel',
               'gehalt_min_eur', 'gehalt_zeitraum', 'skills_top3']

GOLD_PATH = Path('../annotation/meine_gold.csv')
gold = pd.read_csv(GOLD_PATH).set_index('id')
gold.index = gold.index.astype(str)

def load_preds(path):
    """Robust gegen fehlende/leere Dateien: gibt dann ein leeres DataFrame
    mit den Schema-Spalten zurueck, statt abzustuerzen."""
    rows, fails = [], 0
    p = Path(path)
    if p.exists():
        with open(p, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                if obj.get('_parse_fail'):
                    fails += 1
                else:
                    rows.append(obj)
    if not rows:
        return pd.DataFrame(columns=SCHEMA_COLS).set_index('id'), fails
    df = pd.DataFrame(rows).rename(columns={'refnr': 'id'}).set_index('id')
    df.index = df.index.astype(str)
    return df, fails

preds_bl, pf_bl = load_preds('../predictions.jsonl')
preds_A,  pf_A  = load_preds('../predictions_iter_A.jsonl')
preds_B,  pf_B  = load_preds('../predictions_iter_B.jsonl')
preds_7b, pf_7b = load_preds('../predictions_7b_full.jsonl')
preds_3b, pf_3b = load_preds('../predictions_3b_full.jsonl')

for name, df in [('Baseline', preds_bl), ('Iter A', preds_A), ('Iter B', preds_B),
                 ('7B full', preds_7b), ('3B full', preds_3b)]:
    status = f'{len(df)} Anzeigen' if len(df) else 'LEER — auf dem Hub nachlaufen'
    print(f'{name:<10}: {status}')

<jemalloc>: Unsupported system page size


Baseline  : 12 Anzeigen
Iter A    : 12 Anzeigen
Iter B    : 12 Anzeigen
7B full   : 32 Anzeigen
3B full   : 31 Anzeigen


In [2]:
import math

def skill_set(v):
    """Skills robust zu einem Set normalisieren: Liste oder Pipe-String,
    leere/NaN-Werte werden zu einem leeren Set."""
    if v is None:
        return set()
    if isinstance(v, float) and math.isnan(v):
        return set()
    items = v if isinstance(v, list) else str(v).split('|')
    return set(s for s in (str(x).strip().lower() for x in items)
               if s not in ('', 'nan', 'none'))

FIELDS = ['homeoffice', 'vertragsart', 'erfahrungslevel',
          'gehalt_min_eur', 'gehalt_zeitraum', 'skills_top3']

def per_field_accuracy(gold_df, pred_df):
    common = gold_df.index.intersection(pred_df.index)
    if len(common) == 0:
        return None, common
    results = {}
    for field in ['homeoffice', 'vertragsart', 'erfahrungslevel', 'gehalt_zeitraum']:
        g = gold_df.loc[common, field].fillna('').astype(str)
        p = (pred_df.loc[common, field].fillna('').astype(str)
             if field in pred_df.columns else pd.Series('', index=common))
        n_ok = (g == p).sum()
        results[field] = (n_ok, len(common), n_ok / len(common))
    # gehalt_min_eur numerisch
    g = gold_df.loc[common, 'gehalt_min_eur'].fillna(-1).astype(float)
    p = (pred_df.loc[common, 'gehalt_min_eur'].fillna(-1).astype(float)
         if 'gehalt_min_eur' in pred_df.columns else pd.Series(-1.0, index=common))
    n_ok = (g == p).sum()
    results['gehalt_min_eur'] = (n_ok, len(common), n_ok / len(common))
    # skills_top3 reihenfolge-unabhaengig (Set-Match)
    n_ok = 0
    for rid in common:
        g_set = skill_set(gold_df.loc[rid, 'skills_top3'])
        p_set = skill_set(pred_df.loc[rid, 'skills_top3'] if 'skills_top3' in pred_df.columns else '')
        n_ok += int(g_set == p_set)
    results['skills_top3'] = (n_ok, len(common), n_ok / len(common))
    return {f: results[f] for f in FIELDS}, common

def print_table(label, results, common):
    print(f'=== {label} — n={len(common)} ===')
    if results is None:
        print('  (keine Predictions — Datei leer; auf dem Hub nachlaufen)\n')
        return
    print(f'{"Feld":<22} {"korrekt":>8} {"gesamt":>8} {"Accuracy":>10}')
    print('-' * 52)
    for field, (n_ok, n_tot, acc) in results.items():
        note = '  (Set-Match)' if field == 'skills_top3' else ''
        print(f'{field:<22} {n_ok:>8} {n_tot:>8} {acc:>10.1%}{note}')
    print()

bl_res, bl_common = per_field_accuracy(gold, preds_bl)
print_table('Baseline', bl_res, bl_common)

=== Baseline — n=12 ===
Feld                    korrekt   gesamt   Accuracy
----------------------------------------------------
homeoffice                   12       12     100.0%
vertragsart                  10       12      83.3%
erfahrungslevel               7       12      58.3%
gehalt_min_eur               12       12     100.0%
gehalt_zeitraum              12       12     100.0%
skills_top3                   6       12      50.0%  (Set-Match)



In [3]:
# Fehleranalyse auf den 12 Gold-Anzeigen (Iteration A; identisch zur Baseline).
# Schwaechste Felder: erfahrungslevel, skills_top3.
A_res_tmp, A_common_tmp = per_field_accuracy(gold, preds_A)

for field in ['erfahrungslevel', 'skills_top3']:
    print(f'=== Fehleranalyse {field} (Iteration A) ===')
    for rid in A_common_tmp:
        if field == 'skills_top3':
            g_set = skill_set(gold.loc[rid, field])
            p_set = skill_set(preds_A.loc[rid, field] if field in preds_A.columns else '')
            if g_set != p_set:
                print(f'  {rid}: gold={sorted(g_set)}  pred={sorted(p_set)}')
        else:
            g = str(gold.loc[rid, field] or '')
            p = str(preds_A.loc[rid, field] if field in preds_A.columns else '')
            if g != p:
                print(f'  {rid}: gold={g!r}  pred={p!r}')
    print()

=== Fehleranalyse erfahrungslevel (Iteration A) ===
  10000-1001-S: gold='junior'  pred='nicht_genannt'
  10000-1004-S: gold='nicht_genannt'  pred='junior'
  10000-1006-S: gold='nicht_genannt'  pred='egal'
  10000-1011-S: gold='junior'  pred='egal'
  10000-1012-S: gold='nicht_genannt'  pred='egal'

=== Fehleranalyse skills_top3 (Iteration A) ===
  10000-1003-S: gold=['aws', 'python', 'tensorflow']  pred=['python', 'scikit-learn', 'tensorflow']
  10000-1004-S: gold=[]  pred=['pandas', 'python', 'tableau']
  10000-1006-S: gold=['jupyter', 'python', 'sql']  pred=['jupyter notebooks', 'python', 'sql']
  10000-1009-S: gold=[]  pred=['python', 'r', 'sql']
  10000-1011-S: gold=['sql']  pred=['kundenberatung', 'netzwerktechnik', 'sql']
  10000-1012-S: gold=[]  pred=['power_bi', 'python', 'sql']



**Zwei schwächste Felder (Baseline) + Fehler-Hypothesen:**

Schwächste Felder in der Baseline: `erfahrungslevel` (58 %) und `skills_top3` (50 %, Set-Match). (Baseline und Iteration A sind identisch — siehe Δ-Auswertung unten —, die Fehleranalyse oben läuft auf `preds_A`, die Fälle sind also dieselben.)

**Feld 1: `erfahrungslevel` (58 %)** — Diagnose: **Schema-/Prompt-Problem.** Das Modell verwechselt systematisch `nicht_genannt`, `junior` und `egal`. Konkrete Fälle: `10000-1004-S` (Werkstudent → Modell `junior`, Gold `nicht_genannt` — genau der Edge Case aus `01_explore`), `10000-1006-S` und `10000-1012-S` (Gold `nicht_genannt` → Modell `egal`), `10000-1001-S` und `10000-1011-S` (Gold `junior` → Modell verfehlt). Die Grenze zwischen „kein Level genannt", „egal/explizit offen" und „junior" ist im Prompt nicht operationalisiert.

**Feld 2: `skills_top3` (50 %, Set-Match)** — gemischte Ursachen: (a) Oberflächenform statt Inhalt — `10000-1006-S`: Gold `jupyter`, Modell `jupyter notebooks` (derselbe Skill, der Set-Match zählt es als Fehler → Synonym-/Normalisierungs-Thema); (b) echte inhaltliche Divergenz beim dritten Skill — `10000-1003-S`: Gold `aws`, Modell `scikit-learn`; (c) Halluzination — `10000-1009-S`: Gold leer (keine Skills annotiert), Modell erfindet `python|r|sql`.

## Phase 4 — Iteration A Auswertung

**Hypothese:** `homeoffice` steigt durch Prompt-Klarstellung ja/teilweise. Erwartung: Δ +8–17 Pt. Andere Felder: keine Änderung.

In [4]:
A_res, A_common = per_field_accuracy(gold, preds_A)
print_table('Iteration A', A_res, A_common)

if bl_res is not None and A_res is not None:
    print('=== Delta Baseline -> Iteration A ===')
    for field in A_res:
        print(f'  {field:<22}: {A_res[field][2] - bl_res[field][2]:+.1%}')
else:
    print('Delta vs. Baseline: noch nicht berechenbar (predictions.jsonl leer).')

=== Iteration A — n=12 ===
Feld                    korrekt   gesamt   Accuracy
----------------------------------------------------
homeoffice                   12       12     100.0%
vertragsart                  10       12      83.3%
erfahrungslevel               7       12      58.3%
gehalt_min_eur               12       12     100.0%
gehalt_zeitraum              12       12     100.0%
skills_top3                   6       12      50.0%  (Set-Match)

=== Delta Baseline -> Iteration A ===
  homeoffice            : +0.0%
  vertragsart           : +0.0%
  erfahrungslevel       : +0.0%
  gehalt_min_eur        : +0.0%
  gehalt_zeitraum       : +0.0%
  skills_top3           : +0.0%


### Auswertung Iteration A (echte Zahlen, n = 12): 
Baseline und Iteration A sind in allen sechs Feldern identisch, Δ = 0 überall. Die Iteration hatte keinen messbaren Effekt — die Klarstellung betraf nur homeoffice ja/teilweise, aber homeoffice lag schon in der Baseline bei 100 %, dort war nichts zu verbessern. Diagnose/Lehre: Ich habe das Ziel falsch gewählt — die tatsächlich schwachen Felder sind erfahrungslevel (58 %) und skills_top3 (50 %), nicht homeoffice. Richtig wäre gewesen, erst zu profilieren und dann das schwächste Feld anzugehen.

## Phase 4 — Iteration B Auswertung

**Hypothese:** `gehalt_min_eur` steigt durch Stundenloehne→null Klarstellung. Erwartung: Δ +8 Pt (1 Anzeige). Übrige Felder unverändert.

In [5]:
B_res, B_common = per_field_accuracy(gold, preds_B)

def cell(res, field):
    return f'{res[field][2]:.1%}' if res is not None else '—'

ref = A_res or B_res
print(f'{"Feld":<22} {"Baseline":>10} {"Iter. A":>10} {"Iter. B":>10}')
print('-' * 56)
for field in ref:
    print(f'{field:<22} {cell(bl_res, field):>10} {cell(A_res, field):>10} {cell(B_res, field):>10}')

Feld                     Baseline    Iter. A    Iter. B
--------------------------------------------------------
homeoffice                 100.0%     100.0%     100.0%
vertragsart                 83.3%      83.3%      83.3%
erfahrungslevel             58.3%      58.3%      66.7%
gehalt_min_eur             100.0%     100.0%      83.3%
gehalt_zeitraum            100.0%     100.0%      83.3%
skills_top3                 50.0%      50.0%      41.7%


**Auswertung Iteration B (echte Zahlen, n = 12 — Vergleich A → B):** Die Hypothese ist **widerlegt.** Die Klarstellung „Stundenlöhne (EUR/h) → `null`" hat `gehalt_min_eur` nicht verbessert, sondern von 100 % auf 83 % **verschlechtert.**

**Mechanismus:** Die Regel hat überkorrigiert und zwei korrekt erkannte *Monats*gehälter mit-genullt — `10000-1001-S` (Gold 1050/Monat: A `1050` ✓ → B `null` ✗) und `10000-1011-S` (Gold 900/Monat: A `900` ✓ → B `null` ✗). Beides sind niedrige Monatsbeträge, die das Modell nach der neuen Instruktion fälschlich wie Stundenangaben behandelt hat. `gehalt_zeitraum` fiel parallel (100 % → 83 %), `skills_top3` leicht (50 % → 42 %). `erfahrungslevel` stieg nebenbei (58 % → 67 %) — nicht Ziel dieser Iteration.

**Diagnose:** Prompt-Problem mit Nebenwirkung — eine zu absolute Regel trifft auch legitime niedrige Monatsangaben. Konsequenz für die nächste Iteration: die Null-Bedingung an den *explizit genannten Zeitraum* koppeln, nicht an die Höhe des Betrags. Lehre: eine Iteration kann eine Metrik auch *senken* — deshalb messe ich Δ über **alle** Felder, nicht nur über das Zielfeld (Regressionskontrolle).

## Phase 6 — Vollständiger 7B-Run + 3B-Halluzinations-Klassen

Der 7B-Lauf über den vollen Korpus (32 Anzeigen) ist vorhanden (`predictions_7b_full.jsonl`, 0 Schema-Verletzungen). Der 3B-Run auf euler ist ebenfalls durch (`predictions_3b_full.jsonl`, 31/32 — 1 Parse-Fail, selbst schon ein Befund). Unten die Fälle, in denen 7B korrekt liegt und 3B abweicht — daraus die drei Halluzinations-Klassen.

In [6]:
import subprocess
r = subprocess.run(
    ['python', 'annotation/validate.py', '--validate-jsonl', 'predictions_7b_full.jsonl'],
    capture_output=True, text=True, cwd='..'
)
print(r.stdout)


JSONL Schema-Check: predictions_7b_full.jsonl
Geprüfte Zeilen: 32
JSON-Parse-Fails: 0
Keine Feld-Verletzungen.



In [7]:
res_7b, common_7b = per_field_accuracy(gold, preds_7b)
print_table('7B Full (auf Gold-Anzeigen)', res_7b, common_7b)

=== 7B Full (auf Gold-Anzeigen) — n=12 ===
Feld                    korrekt   gesamt   Accuracy
----------------------------------------------------
homeoffice                   12       12     100.0%
vertragsart                  10       12      83.3%
erfahrungslevel               8       12      66.7%
gehalt_min_eur               10       12      83.3%
gehalt_zeitraum              10       12      83.3%
skills_top3                   5       12      41.7%  (Set-Match)



In [8]:
# 3B vs. 7B vs. Gold: Faelle, in denen 7B korrekt liegt und 3B abweicht
if len(preds_3b) == 0:
    print('predictions_3b_full.jsonl ist LEER — der 3B-Lauf auf euler steht noch aus.')
    print('Sobald die Datei existiert, listet diese Zelle automatisch die Faelle,')
    print('in denen das 7B-Modell richtig liegt und das 3B-Modell abweicht.')
else:
    common_all = gold.index.intersection(preds_7b.index).intersection(preds_3b.index)
    print(f'Anzeigen mit allen drei Quellen: {len(common_all)}')
    for field in ['homeoffice', 'vertragsart', 'erfahrungslevel']:
        g  = gold.loc[common_all, field].fillna('').astype(str)
        p7 = preds_7b.loc[common_all, field].fillna('').astype(str) if field in preds_7b.columns else pd.Series('', index=common_all)
        p3 = preds_3b.loc[common_all, field].fillna('').astype(str) if field in preds_3b.columns else pd.Series('', index=common_all)
        cands = common_all[(g == p7) & (g != p3)]
        if len(cands):
            print(f'\n--- {field}: 3B weicht ab, 7B korrekt ---')
            for rid in cands:
                print(f'  {rid}: gold={g[rid]!r}  7B={p7[rid]!r}  3B={p3[rid]!r}')

Anzeigen mit allen drei Quellen: 12

--- homeoffice: 3B weicht ab, 7B korrekt ---
  10000-1008-S: gold='ja'  7B='ja'  3B='remote'

--- vertragsart: 3B weicht ab, 7B korrekt ---
  10000-1001-S: gold='ausbildung'  7B='ausbildung'  3B='aufbauend_ausbildung'
  10000-1011-S: gold='ausbildung'  7B='ausbildung'  3B='aufbaustudiengang'

--- erfahrungslevel: 3B weicht ab, 7B korrekt ---
  10000-1001-S: gold='junior'  7B='junior'  3B='nicht_genannt'


### Drei 3B-Halluzinations-Klassen

Belegt aus `predictions_3b_full.jsonl` (3B weicht ab, wo 7B korrekt liegt) und dem Schema-Check.

**Klasse 1 — Erfundene Schema-Werte (Kategorie-Halluzination).** Das 3B bildet bei `vertragsart` Werte, die das Schema nicht kennt: `10000-1001-S` → `aufbauend_ausbildung`, `10000-1011-S` → `aufbaustudiengang` (Gold und 7B jeweils `ausbildung`). Im vollen Lauf zusätzlich `trainee` und drei Schreibvarianten `aufbauend_/aufbauendes_/aufbauende_ausbildung` — das Modell ist nicht mal mit sich selbst konsistent. Das 7B bleibt durchgehend bei den fünf erlaubten Werten.

**Klasse 2 — Wert-Verwechslung im Schema (über-/unter-interpretiert).** Das 3B wählt einen erlaubten, aber falschen Wert: `10000-1008-S` `homeoffice` → `remote` statt `ja` (liest aus „HO angeboten" volle Ortsunabhängigkeit), `10000-1001-S` `erfahrungslevel` → `nicht_genannt` statt `junior` (übersieht das Signal). 7B liegt bei beiden richtig.

**Klasse 3 — Struktur-/Vollständigkeitsfehler.** Verstöße gegen die Form statt den Inhalt: `skills_top3` mit 4 statt max. 3 Einträgen (2×), leere Pflichtfelder (`homeoffice`/`vertragsart`/`erfahrungslevel` je 1×) und eine Anzeige ganz ohne gültiges JSON (Parse-Fail → 31 statt 32). Das 7B hatte 0 Parse-Fails und 0 Schema-Verletzungen.

**Fazit:** Das kleinere 3B liefert plausibel klingenden Output, aber mit schwächerer Schema- und Strukturtreue — genau der Trade-off, der im Make-or-Buy-Memo gegen das billigere Modell spricht.